# MvDEC 2025: three public datasets, seeds 42-44

Runs the preregistered paper-resolved P0 protocol on REUTERS-10K, 20NEWS, and RCV1-10K. Results are reported as all individual seeds plus mean/sample standard deviation. Best-of-run values are sensitivity diagnostics only. Air Pollution is intentionally excluded.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle/Colab GPU before running this notebook."


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path("/kaggle/working/mvdec_fpmax")
subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run([
    "git", "clone", "--branch", "dev", "--single-branch",
    "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO),
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run([
    sys.executable, str(REPO / "scripts/fetch_dekm_release.py"),
    "--datasets", "REUTERS", "20NEWS", "RCV1",
], cwd=REPO, check=True)

# The official DEKM RCV1 files contain split indices; cache the source matrix used by their loader.
from sklearn.datasets import fetch_rcv1
fetch_rcv1(download_if_missing=True)
print("Official DEKM files and sklearn RCV1 cache are ready.")


In [ ]:
import os
import traceback
from pipeline.public_reproduction_matrix import run_public_matrix

PROTOCOL = "mvdec_2025_paper_resolved_full_batch_v1"
DATASETS = ("REUTERS", "20NEWS", "RCV1")
SEEDS = (42, 43, 44)
OUTPUT_ROOT = Path("/kaggle/working/mvdec-three-datasets-output")
ARCHIVE_ROOT = Path("/kaggle/working/mvdec-three-datasets-archives")
failures = []

for dataset in DATASETS:
    for seed in SEEDS:
        os.environ["PYTHONHASHSEED"] = str(seed)
        try:
            list(run_public_matrix(
                datasets=[dataset],
                protocols=[PROTOCOL],
                seeds=[seed],
                dataset_root=REPO / "external_repos/DEKM/datasets",
                output_root=OUTPUT_ROOT,
                archive_root=ARCHIVE_ROOT,
                progress_interval=100,
                heartbeat_seconds=300,
                resume=True,
            ))
        except Exception as error:
            failures.append({
                "dataset": dataset,
                "seed": seed,
                "error_type": type(error).__name__,
                "error": str(error),
            })
            traceback.print_exc()

print("Failures:", failures)


In [ ]:
import json
import pandas as pd

runs_path = ARCHIVE_ROOT / "public_mvdec_matrix_runs.csv"
aggregate_path = ARCHIVE_ROOT / "public_mvdec_matrix_aggregate.csv"
if runs_path.exists():
    runs = pd.read_csv(runs_path)
    display(runs[["dataset", "protocol_id", "seed", "acc", "nmi", "runtime_seconds"]])
if aggregate_path.exists():
    aggregate = pd.read_csv(aggregate_path)
    display(aggregate)
(ARCHIVE_ROOT / "failures.json").write_text(json.dumps(failures, indent=2), encoding="utf-8")


In [ ]:
import hashlib
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/mvdec-p0-three-datasets-seeds42-44",
    "zip",
    ARCHIVE_ROOT,
)
digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
print("ZIP:", zip_path)
print("SHA-256:", digest)
